# 🧠 Week 10 Lab — Student Version
## Linear Discriminant Analysis (LDA)

**Objectives:**
- Understand how LDA differs from PCA as a projection method
- Implement Fisher's criterion from scratch
- Apply LDA to all four motor control tasks
- Compare LDA with all previous classifiers (NB, LR, KNN, SVM)
- Explore LDA's assumptions and what happens when they fail (QDA)
- Use LDA as dimensionality reduction and interpret loading vectors

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut
from sklearn.metrics import accuracy_score, silhouette_score

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})
logo = LeaveOneGroupOut()

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload week8_data.pkl

In [ ]:
# Load the dataset
with open('week8_data.pkl', 'rb') as f:
    D = pickle.load(f)

# Unpack
emg_raw = D['X_raw']           # (480, 6) EMG features
neural_rates = D['neural_rates']  # (480, 80) neural firing rates
targets = D['targets']          # 0-7 direction labels
group_binary = (D['labels'] == 'impaired').astype(int)  # 0=healthy, 1=impaired
subjects = D['subjects']        # subject IDs for LOSO
target_angles = D['target_angles']  # 8 angles in radians
neuron_pds = D['neuron_pds']    # preferred directions of 80 neurons
joint_angles = D['joint_angles']  # (480, 2) shoulder & elbow

dir_labels = [f'{int(np.rad2deg(a))}°' for a in target_angles]
DIR_COLORS = plt.cm.hsv(np.linspace(0, 1, 9))[:8]

print(f'Dataset: {emg_raw.shape[0]} trials, {emg_raw.shape[1]} EMG, '
      f'{neural_rates.shape[1]} neurons, {len(np.unique(subjects))} subjects')
print(f'Tasks: 8-direction ({len(np.unique(targets))} classes), '
      f'binary ({np.unique(D["labels"])})')

---

## 🟢 Part 1: PCA vs LDA — Different Objectives (Lecture §1–2)

### Exercise 1.0: Toy example — PCA vs LDA on three elongated classes (Lecture Figure 2)

**Learning objective:** Build intuition on a simple 2D toy dataset before working with the 80D neural data. Three elongated Gaussian classes are arranged so that PCA's maximum-variance direction (PC1) runs *along* the classes, while LDA's maximum-separation direction (LD1) runs *between* them.

⚠️ **Note:** This is a synthetic toy dataset (not the course reaching data). We use it because you can see both the data and the projection directions in 2D — something impossible in 80 dimensions.

In [ ]:
# Exercise 1.0: Toy example — PCA vs LDA on 2D data
# This is a SYNTHETIC toy dataset to build intuition, not the course data.
np.random.seed(42)
n_per = 100
means = [[-2, 1], [0, -1], [2, 1]]  # three class centres
angle = np.pi / 6
R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
cov_base = np.array([[2.0, 0], [0, 0.3]])  # elongated
cov = R @ cov_base @ R.T

toy_X = np.vstack([np.random.multivariate_normal(m, cov, n_per) for m in means])
toy_y = np.repeat([0, 1, 2], n_per)
toy_colors = ['#e74c3c', '#3498db', '#2ecc71']

# TODO: Fit PCA and LDA on toy_X
# pca_toy = PCA(n_components=2).fit(toy_X)
# lda_toy = LinearDiscriminantAnalysis(n_components=2).fit(toy_X, toy_y)

# TODO: Create 3-panel figure:
# (A) Scatter + PC1 arrow — observe it runs along the elongated direction
# (B) Scatter + LD1 arrow — observe it runs perpendicular, between classes
# (C) 1D histograms of both projections — PCA overlaps, LDA separates
# YOUR CODE HERE

### Exercise 1.1: PC1 vs LD1 histograms (Lecture Figure 1)

**Learning objective:** See the difference between PCA's maximum-variance axis and LDA's maximum-separation axis on the same 80D neural data. Quantify the difference with the silhouette score.

In [ ]:
# Exercise 1.1: PC1 vs LD1 histograms
sc_neural = StandardScaler().fit_transform(neural_rates)

# TODO: Fit PCA and LDA on sc_neural
# pca = ...
# lda = ...

# TODO: Project onto PC1 and LD1
# proj_pc1 = sc_neural @ pca.components_[0]
# proj_ld1 = lda.transform(sc_neural)[:, 0]

# TODO: Compute silhouette scores for each 1D projection
# sil_pc1 = silhouette_score(...)
# sil_ld1 = silhouette_score(...)

# TODO: Plot histograms side-by-side, coloured by direction
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
# YOUR CODE HERE

plt.tight_layout()
plt.show()

### Exercise 1.2: The Σ = S_B + S_W decomposition (Lecture §2)

**Learning objective:** Verify the total scatter decomposition by computing S_B and S_W from the data and checking that they sum to Σ_total.

In [ ]:
# Exercise 1.2: Verify Σ_total = S_B + S_W
X = sc_neural.copy()
n, p = X.shape
grand_mean = X.mean(axis=0)

# Total scatter (biased covariance = 1/N version)
Sigma_total = np.cov(X, rowvar=False, bias=True)

# TODO: Compute S_B (between-class scatter)
# For each class k, compute (n_k/n) * outer(class_mean - grand_mean)
S_B = np.zeros((p, p))
# YOUR CODE HERE

# TODO: Compute S_W (within-class scatter)
# For each class k, compute (1/n) * X_centered.T @ X_centered
S_W = np.zeros((p, p))
# YOUR CODE HERE

# Check: should be very close to zero
reconstruction_error = np.linalg.norm(Sigma_total - (S_B + S_W))
print(f'||Σ_total - (S_B + S_W)|| = {reconstruction_error:.2e}')

---

## 🟢 Part 2: Fisher's Criterion from Scratch (Lecture §3)

### Exercise 2.1: Implement Fisher's criterion for 2-class EMG (Lecture Figure 3)

**Learning objective:** Code the Fisher solution w* = S_W^{-1}(μ₁ − μ₂) by hand on the **EMG binary diagnosis** data (6 muscles, healthy vs impaired) and verify it matches sklearn's LDA. We also visualise the class means and Fisher direction in PCA space.

⚠️ **Note:** This exercise uses the **EMG data** (6 muscle features), not the 80D neural data.

In [ ]:
# Exercise 2.1: Fisher's criterion from scratch — EMG binary (healthy vs impaired)
# ── DATA: 6-muscle EMG features, 480 trials, binary labels (healthy/impaired) ──
sc_emg = StandardScaler().fit_transform(emg_raw)
muscle_names = ['BIC', 'TRI', 'AD', 'PD', 'BRD', 'PRO']

# Class means
mu_0 = sc_emg[group_binary == 0].mean(axis=0)  # healthy
mu_1 = sc_emg[group_binary == 1].mean(axis=0)  # impaired

# TODO: Compute within-class scatter S_W for 2-class case
# S_W = X_0_centered.T @ X_0_centered + X_1_centered.T @ X_1_centered
S_W_bin = None  # YOUR CODE HERE

# TODO: Compute Fisher direction w* = S_W^{-1} (μ₁ - μ₂)
# Hint: use np.linalg.solve(S_W_bin, mu_1 - mu_0) instead of explicit inverse
w_fisher = None  # YOUR CODE HERE
w_fisher_norm = w_fisher / np.linalg.norm(w_fisher)

# Compare with sklearn
lda_bin = LinearDiscriminantAnalysis().fit(sc_emg, group_binary)
w_sklearn = lda_bin.scalings_[:, 0]
w_sklearn_norm = w_sklearn / np.linalg.norm(w_sklearn)
cos_sim = np.abs(np.dot(w_fisher_norm, w_sklearn_norm))
print(f'Cosine similarity: {cos_sim:.6f}')

# TODO: Create 4-panel figure:
#   (A) Bar chart of class means for each muscle
#   (B) Scatter EMG data in PCA space, show Fisher and naive direction arrows
#   (C) Histogram of Fisher projection for healthy vs impaired
#   (D) Bar chart of Fisher weights for each muscle
# YOUR CODE HERE

### Exercise 2.2: Geometric intuition — naive vs Fisher (Lecture Figure 4)

**Learning objective:** See why projecting onto the line between class means fails when classes are elongated, and how Fisher fixes it.

⚠️ **Note:** Like Exercise 1.0, this uses a **synthetic toy dataset** (two 2D Gaussian classes with shared elongated covariance). We use toy data so you can *see* both projection directions in the same plot — impossible with the 6D EMG or 80D neural data. The lesson carries over: Fisher accounts for within-class scatter, the naive projection doesn't.

In [ ]:
# Exercise 2.2: Naive vs Fisher on toy 2D data
# ── SYNTHETIC TOY DATA (not the course dataset) ──
# Two elongated Gaussian classes where the naive direction fails.
np.random.seed(7)
n_toy = 80
cov_toy = np.array([[3.0, 2.2], [2.2, 2.0]])
X_A = np.random.multivariate_normal([1, 3], cov_toy, n_toy)
X_B = np.random.multivariate_normal([3, 1], cov_toy, n_toy)
X_toy = np.vstack([X_A, X_B])
y_toy = np.array([0]*n_toy + [1]*n_toy)

# TODO: Compute the naive direction (line between means) and normalise it
# w_naive_n = ...

# TODO: Compute the Fisher direction using sklearn LDA and normalise it
# w_lda_n = ...

# TODO: Create 3-panel figure:
# (A) Scatter plot with both direction arrows
# (B) Histogram of naive projection — observe overlap
# (C) Histogram of Fisher projection — observe separation
# YOUR CODE HERE

---

## 🟡 Part 3: LDA on All Four Tasks (Lecture §4)

### Exercise 3.1: Six-method comparison — LOSO (Lecture Figure 5)

**Learning objective:** Add LDA to the running comparison table from previous weeks and see where it excels and where it doesn't.

In [ ]:
# Exercise 3.1: Six-method LOSO comparison
tasks = [
    ('EMG Binary', emg_raw, group_binary),
    ('Neural Binary', neural_rates, group_binary),
    ('EMG Direction', emg_raw, targets),
    ('Neural Direction', neural_rates, targets),
]

methods = [
    ('Naive Bayes', GaussianNB()),
    ('Log. Reg.', LogisticRegression(C=10, max_iter=2000)),
    ('KNN (k=5)', KNeighborsClassifier(n_neighbors=5)),
    ('Lin. SVM', SVC(kernel='linear', C=1)),
    ('RBF SVM', SVC(kernel='rbf', C=10, gamma='scale')),
    # TODO: Add LDA to this list
]

results = np.zeros((len(methods), len(tasks)))

# TODO: Loop over tasks and methods, compute LOSO accuracy
# Use Pipeline([('s', StandardScaler()), ('clf', clf)])
# YOUR CODE HERE

# TODO: Plot in two-panel layout (matching Lecture Figure 5):
#   Left panel: Direction decoding (EMG + Neural bars side-by-side)
#   Right panel: Binary diagnosis (EMG + Neural bars side-by-side)
# Highlight LDA bars with orange edge
# YOUR CODE HERE

### Exercise 3.2: LDA component sweep (Lecture Figure 6)

**Learning objective:** Discover how many LDA components are needed. LDA can have at most K−1 = 7 components for 8 classes. Does accuracy plateau before 7?

In [ ]:
# Exercise 3.2: Component sweep — LDA vs PCA
sc_neural = StandardScaler().fit_transform(neural_rates)
n_comps = range(1, 8)  # 1 to 7 components

acc_lda = []
acc_pca = []

for nc in n_comps:
    # TODO: Build Pipeline with LDA(n_components=nc) + KNN(5)
    # Compute LOSO accuracy and append to acc_lda
    pass  # YOUR CODE HERE

    # TODO: Build Pipeline with PCA(n_components=nc) + KNN(5)
    # Compute LOSO accuracy and append to acc_pca
    pass  # YOUR CODE HERE

# TODO: Plot both curves on the same axes
# YOUR CODE HERE

---

## 🟡 Part 4: LDA as Dimensionality Reduction (Lecture §5)

### Exercise 4.1: PCA vs LDA 2D scatter (Lecture Figure 7)

**Learning objective:** Visualise PCA and LDA projections side-by-side for both feature sets. Use silhouette scores to quantify the difference.

In [ ]:
# Exercise 4.1: 2×2 PCA vs LDA scatter
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# TODO: For each feature set (neural, EMG):
#   - Compute PCA(2) and LDA(2) projections
#   - Scatter plot coloured by direction
#   - Compute and display silhouette score in each title
# YOUR CODE HERE

plt.tight_layout()
plt.show()

### Exercise 4.2: LDA loadings recover cosine tuning (Lecture Figure 8)

**Learning objective:** Show that LDA's LD1 and LD2 weight vectors correspond to the horizontal and vertical axes of the directional tuning circle.

In [ ]:
# Exercise 4.2: LDA loadings coloured by neuron preferred direction
lda_neural = LinearDiscriminantAnalysis().fit(sc_neural, targets)
loadings = lda_neural.scalings_  # (80, 7)

# TODO: Plot LD1 and LD2 loadings as bar charts, coloured by neuron PD
# Hint: pd_colors = plt.cm.hsv(neuron_pds / (2 * np.pi))
# YOUR CODE HERE

# TODO: Compute correlation between LD1 weights and cos(PD),
#        and between LD2 weights and sin(PD)
# YOUR CODE HERE

---

## 🔴 Part 5: Testing Assumptions — LDA vs QDA (Lecture §6–7)

### Exercise 5.1: Covariance ellipses (Lecture Figure 9)

**Learning objective:** Visualise the equal-covariance assumption. Do the 8 direction classes really share the same spread? The left panel shows 8 directions (healthy only) — if ellipses are similar, the shared-covariance assumption holds. The right panel shows healthy vs impaired — if ellipses differ, LDA's assumption is violated and QDA may help.

In [ ]:
# Exercise 5.1: Covariance ellipses in neural PCA space
from matplotlib.patches import Ellipse

# Helper: plot covariance ellipse
def plot_cov_ellipse(ax, mean, cov, n_std=1.5, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]  # sort descending
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(ell)

pca_neur_2d = PCA(n_components=2).fit_transform(sc_neural)

# TODO: Create 2-panel figure:
# Panel A: 8 directions (healthy only) with covariance ellipses
#   → if ellipses are similar, equal-covariance assumption holds
# Panel B: Healthy vs Impaired with covariance ellipses
#   → if ellipses differ, equal-covariance assumption is violated
# YOUR CODE HERE

### Exercise 5.2: LDA vs QDA (Lecture Figure 10)

**Learning objective:** See when relaxing the equal-covariance assumption helps (EMG binary) and when it causes catastrophic failure (neural direction).

**Expected results:**
- EMG Binary: QDA **improves** over LDA (~87.5% vs 78.5%) because the two classes genuinely have different covariance structures.
- Neural Direction: QDA **crashes** (0%) because estimating eight 80×80 covariance matrices from only 60 trials per class is mathematically impossible (n < p).

In [ ]:
# Exercise 5.2: LDA vs QDA comparison
#
# QDA estimates a SEPARATE covariance matrix for each class.
# EMG binary: 2 classes × 6 features → 21 params each, from 240 trials → fine
# Neural dir: 8 classes × 80 features → 3,240 params each, from 60 trials → IMPOSSIBLE
#
# Expected results:
#   EMG Binary: QDA ~87.5% (improves over LDA's 78.5%)
#   Neural Dir: QDA 0% (crashes — the covariance matrices are singular)
#
# TODO: Evaluate LDA and QDA on both tasks
# Use QDA() without reg_param — let the crash happen naturally
# Wrap in try/except to catch the crash and set acc = 0
# Plot a bar chart matching the lecture figure (hatched bar for the crash)
# YOUR CODE HERE

### 🤔 Thought Exercise
QDA needs to estimate K separate covariance matrices. EMG binary has 2 classes × 6 features = 21 unique parameters per matrix, estimated from 240 trials. Neural direction has 8 classes × 80 features = 3,240 unique parameters per matrix, estimated from 60 trials.

**Question:** What ratio of trials-to-parameters would you want as a minimum for reliable covariance estimation? (Think about this in terms of the bias-variance tradeoff.)

---

## 🔴 Part 6: Bringing It Together (Lecture §8)

### Exercise 6.1: Timing comparison (Lecture Figure 11)

**Learning objective:** Benchmark training and prediction time for all 6 methods. LDA's closed-form solution should be among the fastest.

In [ ]:
# Exercise 6.1: Timing comparison
import time

# TODO: Time fit() and predict() for all 6 methods
# Run each 100 times and average for stable estimates
# Plot horizontal bar chart of train time and predict time
# YOUR CODE HERE

### Exercise 6.2: Drift robustness simulation (Lecture Figure 12)

**Learning objective:** Simulate electrode drift (additive noise to neural features) and measure how each method's accuracy degrades.

In [ ]:
# Exercise 6.2: Drift robustness
noise_levels = np.linspace(0, 2.0, 10)

# TODO: For each noise level, add Gaussian noise to the standardized neural data
# Evaluate NB, LR, KNN, LDA, RBF SVM at each noise level
# Plot accuracy vs noise level for all methods
# YOUR CODE HERE

### Exercise 6.3: Full comparison table (Lecture Figure 13)

**Learning objective:** Compile the complete picture — accuracy, timing, hyperparameters, and when to use each method.

In [ ]:
# Exercise 6.3: Summary table
# TODO: Print a table showing all 6 methods × 4 tasks
# Include training speed and hyperparameter count
# Discuss: which method would you choose for a BCI that needs
# to retrain every session with limited data?
# YOUR CODE HERE

### 🤔 Final Thought Exercise
A BCI research team needs a decoder that:
1. Retrains from scratch every session (data changes daily)
2. Uses only 2 minutes of calibration data (~60 trials)
3. Has to work reliably without manual tuning

Based on what you've learned in Weeks 5–10, which classifier would you recommend and why? Consider accuracy, training speed, hyperparameter sensitivity, and data efficiency.

---

## Summary

In this lab you:

1. **Compared PCA and LDA** as projection methods and measured the difference with silhouette scores
2. **Verified Σ = S_B + S_W** — the decomposition that connects unsupervised and supervised projection
3. **Implemented Fisher's criterion** from scratch and matched sklearn's result
4. **Evaluated LDA** on all four motor control tasks alongside five previous methods
5. **Discovered** that 2–3 LDA components capture most of the 8-direction structure
6. **Interpreted LDA loadings** and found that they recover the cosine tuning structure
7. **Tested the equal-covariance assumption** and saw QDA help (EMG) vs crash (neural)
8. **Benchmarked** computational cost and drift robustness

**Key insight:** LDA's strength is being the right tool for the most common case — Gaussian-ish classes with similar spread, moderate data, and no time to tune hyperparameters.